In [1]:
from faster_whisper import WhisperModel
import os
import json

C:\Users\ASHOKA MS\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = WhisperModel("base", compute_type="int8")


In [3]:
audio_folder = "datasets/clean_audio"

text_output = "datasets/asr_text"
json_output = "datasets/asr_json"

os.makedirs(text_output, exist_ok=True)
os.makedirs(json_output, exist_ok=True)


In [ ]:
for file in os.listdir(audio_folder):
    if file.endswith(".wav"):
        path = os.path.join(audio_folder, file)

        print("Transcribing:", file)

        segments, info = model.transcribe(path)

        transcript = ""
        segment_data = []

        for segment in segments:
            transcript += segment.text + " "
            segment_data.append({
                "start": segment.start,
                "end": segment.end,
                "text": segment.text
            })

        name = os.path.splitext(file)[0]

        # Save text transcript
        with open(os.path.join(text_output, name + ".txt"),
                  "w", encoding="utf-8") as f:
            f.write(transcript.strip())

        # Save timestamps JSON
        with open(os.path.join(json_output, name + ".json"),
                  "w", encoding="utf-8") as f:
            json.dump(segment_data, f, indent=4)

print(" Transcription complete!")


Transcribing: audio1.wav



KeyboardInterrupt


KeyboardInterrupt



In [6]:
from difflib import SequenceMatcher

gt_folder = "datasets/transcripts"
pred_folder = text_output

scores = []

for file in os.listdir(gt_folder):
    if file.endswith(".txt") and file in os.listdir(pred_folder):

        gt = open(os.path.join(gt_folder, file), encoding="utf-8").read()
        pred = open(os.path.join(pred_folder, file), encoding="utf-8").read()

        score = SequenceMatcher(None, gt, pred).ratio()
        scores.append(score)

print("Average Similarity:", round(sum(scores)/len(scores),3))


Average Similarity: 0.199


In [7]:
from jiwer import wer

error_rates = []

for file in os.listdir(gt_folder):
    if file.endswith(".txt") and file in os.listdir(pred_folder):

        gt = open(os.path.join(gt_folder, file), encoding="utf-8").read()
        pred = open(os.path.join(pred_folder, file), encoding="utf-8").read()

        error_rates.append(wer(gt, pred))

print("Average WER:", round(sum(error_rates)/len(error_rates),3))

Average WER: 0.386
